# **Bitcoin Cycles**

* * *

This notebook conducts a detailed exploration of Bitcoin price cycles, focusing on identifying patterns related to top/bottoms phases and halving events. 

Bitcoin halving, a significant event that reduces the Bitcoin mining reward by half approximately every four years, is often associated with fluctuations in Bitcoin's price. This analysis aims to determine whether there is a cyclical pattern in Bitcoin prices that correlates with these halving events.

### Introduction

The behavior of Bitcoin’s price showcases cyclical trends driven by market dynamics, investor sentiment, and pivotal events such as halvings. These halvings, predictable in their occurrence, theoretically influence supply and demand dynamics, potentially leading to significant price cycles. By examining the synchronization between Bitcoin's price cycles and halving events, this analysis seeks to uncover recurring patterns that could inform investment strategies and market predictions.

### Notebook Structure

The notebook is organized into several sections to facilitate a structured analysis:

1.  **Importing Libraries:** Set up the environment by importing essential Python libraries for data processing and visualization.
2.  **Function Definitions:** Define helper functions to aid in data processing, visualization, and annotating significant events.
3.  **Data Preparation:** Load and preprocess Bitcoin price data to align with the analysis requirements.
4.  **Cycle Visualization:** Graphically represent the Bitcoin price cycles, highlighting key phases and aligning them with halving events.
5.  **Conclusion:** Draw conclusions from the visual and statistical analysis, offering insights into the cyclical nature of Bitcoin prices.

In [ ]:
import pandas as pd
import sys
import sys, os
# Force data saving in the existing local directory
_base = os.getcwd()
while _base and not os.path.isdir(os.path.join(_base, 'functions')):
    _parent = os.path.dirname(_base)
    if _parent == _base: break
    _base = _parent
if os.path.abspath(_base) not in sys.path:
    sys.path.insert(0, os.path.abspath(_base))
from functions.functions_cycles import plot_bitcoin_cycles, manipulation
from functions.fetch_data import fetch_crypto_data

btc_data = fetch_crypto_data("btc")

# Data Manipulation
---------


### Defining Bitcoin Event Dates

To analyze the cyclical nature of Bitcoin prices effectively, I begin by marking out the dates of significant events, including halvings and notable peaks and bottoms in past price cycles:

*   **Halving Dates:**
    
    *   November 28, 2012
    *   July 9, 2016
    *   May 11, 2020
    *   April 19, 2024
*   **Peak Price Dates:**
    
    *   November 30, 2013
    *   December 17, 2017
    *   November 10, 2021
    *   October 6, 2025
*   **Bottom Price Dates:**
    
    *   January 14, 2015
    *   December 15, 2018
    *   November 21, 2022
    *   July 1, 2026

These dates represent critical milestones in the Bitcoin market, with halvings significantly influencing the supply of new bitcoins, potentially affecting their price.

### The cycle clock

Every cycle has the same length. The period and the phase are fitted by least squares on all the tops *and* bottoms above together, with each bottom a quarter of a cycle after its top (left free, the fit puts it at 0.249, so the quarter is kept exact). The best period is **1420 days**.

Before, the period was the plain average of the top-to-top intervals and the phase was counted from the first day of data. That clock ignored the bottoms entirely, and it put the July 2026 bottom 104 days late. Fitting tops and bottoms together brings the worst miss down to 65 days - still the latest bottom, because the 2026 bear market was unusually short (268 days, against 363-410 before), and no single fixed period can land on it without missing everything else.

### Output from Data Manipulation

The manipulation function is designed to process Bitcoin price data to extract and calculate various cycle-related parameters. The output includes:

*   `cycle_dates`: A series of datetime objects outlining the analyzed time frame.
*   `cycle_wave`: A numerical array indicating the cyclical pattern in the data.
*   `next_peak_prediction_lower`: The predicted lower bound date for the next price peak.
*   `next_peak_prediction_upper`: The predicted upper bound date for the next price peak.
*   `next_peak_prediction`: The central predicted date for the next peak.
*   `sin_derivative`: A numerical array showing the derivative of the cyclical wave, useful for identifying trend changes.
*   `zero_crossings`: Indices where the cycle wave crosses zero, indicating potential trend reversals.
*   `current_cycle_percentage`: The current phase of the cycle as a percentage, based on the most recent price data.
*   `sin_maxima`: Indices where the cycle reaches local maxima, typically aligning with peak prices.
*   `sin_minima`: Indices where the cycle reaches local minima, often corresponding to price bottoms.

These parameters are crucial for the subsequent analysis and visualization sections, aiding in the understanding of Bitcoin's price dynamics in relation to its cyclical events.

In [ ]:
halving_dates = pd.to_datetime(["2012-11-28", "2016-07-09", "2020-05-11", "2024-04-19"])
tops_dates = pd.to_datetime(["2013-11-30", "2017-12-17", "2021-11-10", "2025-10-06"])
bottoms_dates = pd.to_datetime(["2015-01-14", "2018-12-15", "2022-11-21", "2026-07-01"])

# The manipulation function returns several outputs related to cycle analysis
cycle_dates,cycle_wave,next_peak_prediction_lower,next_peak_prediction_upper,next_peak_prediction,sin_derivative,zero_crossings,current,sin_maxima,sin_minima, = manipulation(btc_data, tops_dates, bottoms_dates)

# Plot Bitcoin Cycles
----------------

This section focuses on graphically representing Bitcoin price cycles to identify and understand the cyclical patterns and key phases within the market. The visualization includes several components to enhance clarity and provide a comprehensive analysis:

### Plot Components

*   **Bitcoin Price Data:** The primary plot displays Bitcoin prices on a logarithmic scale, which is essential for viewing exponential price movements clearly over time.
    
*   **Annotations for Key Events:** The plot is annotated to highlight significant events including halvings, price tops, and bottoms. These dates are marked to show their impact and correlation with the price cycles.
    
*   **Cyclical Pattern Overlay:** A sinusoidal curve is overlaid on the price data to depict the underlying cyclical pattern. This curve helps illustrate the rhythmic nature of the price movements and is essential for identifying cyclical trends.
    
*   **Cycle Phase Highlighting:** Different phases of the cycle, such as maxima, minima, and zero crossings, are clearly annotated. This includes visual cues like points or lines at these critical junctures, providing insights into potential turning points in the market.
    
*   **Phases Fill:** The areas between different phases are shaded to differentiate between the rising and falling stages of the cycle, enhancing the visual understanding of market dynamics.
    


In [ ]:
plot_bitcoin_cycles(
    btc_data,
    halving_dates,
    tops_dates,
    bottoms_dates,
    None,  # next_peak_prediction is no longer used
    None,  # next_peak_prediction_lower is no longer used
    None,  # next_peak_prediction_upper is no longer used
    cycle_dates,
    cycle_wave,
    sin_derivative,
    zero_crossings,
    sin_minima,
    sin_maxima,
    current
)

# Alternative: Asymmetric Cycle Wave
----------------

The chart above uses a symmetric sine for the cyclical pattern (bull leg and bear leg each exactly half the cycle). Historically that's not how BTC cycles behave: the climb from bottom to top takes roughly 3/4 of a cycle (~3 years), the drop from top to bottom only ~1/4 (~1 year).

This version reshapes the wave to match that rhythm - still a smooth curve (no kinks), built from two cosine arcs joined at their flat (zero-slope) points - so the wave's own maximum lands on the top and its own minimum lands on the bottom, instead of the top landing on the sine's peak while the bottom only approximates a zero-crossing. It runs on exactly the same fitted clock as the sine, so every cycle has the same length.

Each cycle runs from one bottom of the clock to the next and is split into 4 equal time steps (1, 2, 3, 4): the top always falls on step 3, the bottom always on step 4. The *current cycle position* box counts from the last confirmed bottom: the share of a cycle elapsed since then.

In [ ]:
from functions.functions_cycles_asym import manipulation_asym, plot_bitcoin_cycles_asym

(
    cycle_dates_asym, cycle_wave_asym, _npl, _npu, _np,
    sin_derivative_asym, zero_crossings_asym, current_asym,
    sin_maxima_asym, sin_minima_asym,
) = manipulation_asym(btc_data, tops_dates, bottoms_dates)

plot_bitcoin_cycles_asym(
    btc_data,
    halving_dates,
    tops_dates,
    bottoms_dates,
    None,
    None,
    None,
    cycle_dates_asym,
    cycle_wave_asym,
    sin_derivative_asym,
    zero_crossings_asym,
    sin_minima_asym,
    sin_maxima_asym,
    current_asym,
)

# Conclusions
------------

The exploration of Bitcoin price cycles offers essential insights into the market's behavior, providing a basis for informed investment strategies. Here are the critical findings from this analysis:

1.  **Cyclical Nature of Bitcoin Prices:** This analysis validates the cyclical patterns within Bitcoin prices, clearly influenced by halving events. This periodic nature underscores the predictability in the market dynamics around these events.
    
2.  **Validation of Hypotheses:** The data supports the hypothesis that the peaks of previous Bitcoin cycles align with the 50% phase of the sinusoidal model, and the bottoms coincide with the 75% phase. This confirmation lends credence to the underlying assumptions of our cyclical analysis.
    
3.  **Predictive Insights:** The cyclical model's projection held up well: the top the 2021 peak implied arrived on October 6, 2025, and the bottom that follows it is confirmed at July 1, 2026 - after a bear market of only 268 days, the shortest yet, which lands two months ahead of the clock. On the fitted 1420-day clock, the next peak would be expected around August 2029. These predictions are invaluable for market participants looking to time their investment decisions effectively.
    

### Summary

This detailed analysis not only reaffirms the cyclical nature of Bitcoin prices but also highlights the predictive power of understanding these cycles in relation to key events like halvings. By recognizing these patterns, investors and analysts can better navigate the volatile cryptocurrency market, planning their strategies with greater confidence based on anticipated price movements. 

Not a financial advice, just research.